# PREPROCESSING 

Scope is to do cleaning , target encoding , and categorical feature encoding

In [3]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.preprocessing import LabelEncoder ,OneHotEncoder

# --- Constants ---
DATA_DIR = Path("../data")
TRAIN_CSV = DATA_DIR / "train.csv"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = Path("../models")

ID_COLUMN = "id"
TARGET_COLUMN = "class"
NUMERIC_FEATURES = ["alpha", "delta", "u", "g", "r", "i", "z", "redshift"]
CATEGORICAL_FEATURES = ["spectral_type", "galaxy_population"]

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

1. defensive validation


In [4]:
train_df = pd.read_csv(TRAIN_CSV)

required_columns = {ID_COLUMN, TARGET_COLUMN, *NUMERIC_FEATURES, *CATEGORICAL_FEATURES}
missing = required_columns - set(train_df.columns)
assert not missing, f"train.csv is missing expected columns: {missing}"
assert train_df[ID_COLUMN].is_unique, "Duplicate ids found in train.csv"
assert train_df.isnull().sum().sum() == 0, "Unexpected missing values found"

print(f"Loaded {train_df.shape[0]:,} rows, {train_df.shape[1]} columns — all checks passed")

Loaded 577,347 rows, 12 columns — all checks passed


2. Dtype cleanup

- casting the two categorical columns to pandas category dtype reduces memory


In [5]:
mem_before = train_df.memory_usage(deep=True).sum() / 1e6

for col in CATEGORICAL_FEATURES:
    train_df[col] = train_df[col].astype("category")

mem_after = train_df.memory_usage(deep=True).sum() / 1e6
print(f"Memory usage: {mem_before:.1f} MB -> {mem_after:.1f} MB")

Memory usage: 150.8 MB -> 78.6 MB


3. Target Encoding


In [6]:
label_encoder = LabelEncoder()
train_df["class_encoded"] = label_encoder.fit_transform(train_df[TARGET_COLUMN])

class_mapping = dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))
print("Label mapping:", class_mapping)

joblib.dump(label_encoder, MODELS_DIR / "label_encoder.joblib")
print(f"Saved label encoder to {MODELS_DIR / 'label_encoder.joblib'}")

Label mapping: {'GALAXY': np.int64(0), 'QSO': np.int64(1), 'STAR': np.int64(2)}
Saved label encoder to ..\models\label_encoder.joblib


4. Categorical encoding

- doing OHE for spectral_type and galaxy_population
- **`handle_unknown="ignore"`**: if the test set ever contains a category not seen in training (shouldn't happen here since both columns are small closed taxonomies, but this is defensive, production-style coding rather than assuming clean data forever), the encoder emits an all-zero row instead of crashing at inference time.
- **Fitting the encoder on train categories is *not* a leakage risk here**, unlike fitting a `StandardScaler` would be. The categories (`M`, `O/B`, `G/K`, `A/F`, `Red_Sequence`, `Blue_Cloud`) are a fixed physical taxonomy independent of the target — the risk with `StandardScaler` is that its mean/std are computed *from the data distribution itself*, which differs slightly across CV folds. One-hot categories aren't statistics of the data, so there's nothing to leak.

In [7]:
onehot_encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoded_array = onehot_encoder.fit_transform(train_df[CATEGORICAL_FEATURES])
encoded_columns = onehot_encoder.get_feature_names_out(CATEGORICAL_FEATURES)
encoded_df = pd.DataFrame(encoded_array, columns=encoded_columns, index=train_df.index)

joblib.dump(onehot_encoder, MODELS_DIR / "onehot_encoder.joblib")
print(f"Saved one-hot encoder to {MODELS_DIR / 'onehot_encoder.joblib'}")
encoded_df.head()

Saved one-hot encoder to ..\models\onehot_encoder.joblib


,spectral_type_A/F,spectral_type_G/K,spectral_type_M,spectral_type_O/B,galaxy_population_Blue_Cloud,galaxy_population_Red_Sequence
0,0.0,0.0,1.0,0.0,0.0,1.0
1,0.0,0.0,1.0,0.0,0.0,1.0
2,0.0,0.0,0.0,1.0,1.0,0.0
3,0.0,0.0,1.0,0.0,0.0,1.0
4,0.0,0.0,1.0,0.0,0.0,1.0


5. Numerical Feature Scaling - will do later 

6. ASSEMBLE + processed dataset

Combining raw numeric features, one-hot encoded categoricals, and the encoded target into a single processed file that every downstream notebook loads directly — so the encoding logic in this notebook never has to be re-run or duplicated elsewhere.

In [8]:
processed_df = pd.concat(
    [
        train_df[[ID_COLUMN, *NUMERIC_FEATURES]],
        encoded_df,
        train_df[[TARGET_COLUMN, "class_encoded"]],
    ],
    axis=1,
)

assert processed_df.shape[0] == train_df.shape[0], "Row count changed during processing"
assert processed_df.isnull().sum().sum() == 0, "Nulls introduced during processing"

output_path = PROCESSED_DIR / "train_processed.csv"
processed_df.to_csv(output_path, index=False)
print(f"Saved {processed_df.shape[0]:,} rows, {processed_df.shape[1]} columns to {output_path}")
processed_df.head()

Saved 577,347 rows, 17 columns to ..\data\processed\train_processed.csv


,id,alpha,delta,u,g,r,i,z,redshift,spectral_type_A/F,spectral_type_G/K,spectral_type_M,spectral_type_O/B,galaxy_population_Blue_Cloud,galaxy_population_Red_Sequence,class,class_encoded
0,0,147.734256,16.959273,25.472123,21.895559,20.357926,19.257113,18.621057,0.408982,0.0,0.0,1.0,0.0,0.0,1.0,GALAXY,0
1,1,127.988677,32.346716,20.778509,19.087062,17.587208,17.226067,16.786433,0.157976,0.0,0.0,1.0,0.0,0.0,1.0,GALAXY,0
2,2,179.792648,35.344843,21.035203,21.079128,21.171840,20.582629,20.557366,2.823770,0.0,0.0,0.0,1.0,1.0,0.0,QSO,1
3,3,225.818295,48.569421,23.305056,21.050736,19.017754,18.365658,17.914952,0.536099,0.0,0.0,1.0,0.0,0.0,1.0,GALAXY,0
4,4,141.836135,19.342852,21.703158,19.471680,18.234449,17.899447,17.616185,0.555761,0.0,0.0,1.0,0.0,0.0,1.0,GALAXY,0
